Install the dependencies

In [6]:
#Install Required Libraries
!pip -q install ultralytics supervision

print("=" * 70)
print("Libraries Installed Successfully")
print("=" * 70)

Libraries Installed Successfully


Import the dependencies

In [7]:
# Import Libraries
import os
import random
import zipfile
import warnings

import numpy as np
import pandas as pd

import torch

from ultralytics import YOLO

warnings.filterwarnings("ignore")

print("=" * 70)
print("Libraries Imported Successfully")
print("=" * 70)

Libraries Imported Successfully


In [8]:
#  Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

print("=" * 70)
print("Google Drive Mounted Successfully")
print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive Mounted Successfully


In [9]:
# Extract Dataset

ZIP_FILE = "/content/drive/MyDrive/YOLO/data/Suspicious Behavior.v1-v1.802img-2025-02-12-11-30pm.yolov8.zip"

EXTRACT_DIR = "/content/drive/MyDrive/YOLO/data/SuspiciousBehavior"

if not os.path.exists(EXTRACT_DIR):

    with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:

        zip_ref.extractall(EXTRACT_DIR)

    print("Dataset Extracted Successfully.")

else:

    print("Dataset Already Extracted.")

Dataset Extracted Successfully.


In [10]:
# Random Seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("Random Seed:", SEED)
print("=" * 70)

Random Seed: 42


In [11]:
# GPU Check

print("=" * 70)

print("PyTorch :", torch.__version__)

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():

    DEVICE = 0

    print("GPU :", torch.cuda.get_device_name(0))

else:

    DEVICE = "cpu"

print("=" * 70)

PyTorch : 2.11.0+cu128
CUDA Available : True
GPU : Tesla T4


In [12]:
#  Project Paths

PROJECT_DIR = "/content/drive/MyDrive/YOLO/data/SuspiciousBehavior"

DATA_YAML = os.path.join(PROJECT_DIR, "data.yaml")

RESULT_DIR = os.path.join(PROJECT_DIR, "results")

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")

os.makedirs(RESULT_DIR, exist_ok=True)

os.makedirs(MODEL_DIR, exist_ok=True)

print("=" * 70)
print("Project Path Ready")
print("=" * 70)

print(PROJECT_DIR)

Project Path Ready
/content/drive/MyDrive/YOLO/data/SuspiciousBehavior


In [13]:
# Experiment Configuration

IMAGE_SIZE = 640

BATCH_SIZE = 16

EPOCHS = 50

WORKERS = 2

LEARNING_RATE = 0.001

print("=" * 70)

print("Experiment Configuration")

print("=" * 70)

print(f"Image Size : {IMAGE_SIZE}")

print(f"Batch Size : {BATCH_SIZE}")

print(f"Epochs     : {EPOCHS}")

print(f"Workers    : {WORKERS}")

print("=" * 70)

print("Environment Ready")

Experiment Configuration
Image Size : 640
Batch Size : 16
Epochs     : 50
Workers    : 2
Environment Ready


In [14]:
#  Verify Folder Structure

TRAIN_IMAGES = os.path.join(PROJECT_DIR, "train", "images")
TRAIN_LABELS = os.path.join(PROJECT_DIR, "train", "labels")

VALID_IMAGES = os.path.join(PROJECT_DIR, "valid", "images")
VALID_LABELS = os.path.join(PROJECT_DIR, "valid", "labels")

TEST_IMAGES = os.path.join(PROJECT_DIR, "test", "images")
TEST_LABELS = os.path.join(PROJECT_DIR, "test", "labels")

print("=" * 70)
print("YOLO Dataset Structure")
print("=" * 70)

folders = {
    "Train Images": TRAIN_IMAGES,
    "Train Labels": TRAIN_LABELS,
    "Valid Images": VALID_IMAGES,
    "Valid Labels": VALID_LABELS,
    "Test Images": TEST_IMAGES,
    "Test Labels": TEST_LABELS,
    "data.yaml": DATA_YAML
}

all_exist = True

for name, path in folders.items():

    status = os.path.exists(path)

    print(f"{name:<18}: {'✓ Found' if status else '✗ Missing'}")

    if not status:
        all_exist = False

print("=" * 70)

if all_exist:
    print("Dataset Structure Verified Successfully")
else:
    print("Dataset Structure Incomplete")

YOLO Dataset Structure
Train Images      : ✓ Found
Train Labels      : ✓ Found
Valid Images      : ✓ Found
Valid Labels      : ✓ Found
Test Images       : ✓ Found
Test Labels       : ✓ Found
data.yaml         : ✓ Found
Dataset Structure Verified Successfully


In [15]:
#Verify data.yaml

import yaml

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("=" * 70)
print("data.yaml")
print("=" * 70)

for key, value in config.items():
    print(f"{key} : {value}")

data.yaml
train : ../train/images
val : ../valid/images
test : ../test/images
nc : 3
names : ['Fight', 'Normal', 'Poster']
roboflow : {'workspace': 'lazyzone', 'project': 'suspicious-behavior-fxla5', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/lazyzone/suspicious-behavior-fxla5/dataset/1'}


In [16]:
#  Verify Image-Label Pairs

def count_files(folder, extensions):

    return len([
        f for f in os.listdir(folder)
        if f.lower().endswith(extensions)
    ])

train_img = count_files(TRAIN_IMAGES, (".jpg", ".jpeg", ".png"))
train_lbl = count_files(TRAIN_LABELS, (".txt",))

valid_img = count_files(VALID_IMAGES, (".jpg", ".jpeg", ".png"))
valid_lbl = count_files(VALID_LABELS, (".txt",))

test_img = count_files(TEST_IMAGES, (".jpg", ".jpeg", ".png"))
test_lbl = count_files(TEST_LABELS, (".txt",))

summary = pd.DataFrame({

    "Split":["Train","Validation","Test"],

    "Images":[train_img, valid_img, test_img],

    "Labels":[train_lbl, valid_lbl, test_lbl]

})

display(summary)

,Split,Images,Labels
0,Train,1688,1688
1,Validation,161,161
2,Test,80,80


In [17]:
#  Verify Classes

print("=" * 70)
print("Class Names")
print("=" * 70)

names = config["names"]

if isinstance(names, dict):
    for idx, cls in names.items():
        print(f"{idx} -> {cls}")
else:
    for idx, cls in enumerate(names):
        print(f"{idx} -> {cls}")

print("=" * 70)

Class Names
0 -> Fight
1 -> Normal
2 -> Poster


In [18]:
# Dataset Summary
TOTAL_IMAGES = train_img + valid_img + test_img

TOTAL_LABELS = train_lbl + valid_lbl + test_lbl

print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(f"Training Images    : {train_img}")
print(f"Validation Images  : {valid_img}")
print(f"Testing Images     : {test_img}")

print()

print(f"Training Labels    : {train_lbl}")
print(f"Validation Labels  : {valid_lbl}")
print(f"Testing Labels     : {test_lbl}")

print()

print(f"Total Images       : {TOTAL_IMAGES}")
print(f"Total Labels       : {TOTAL_LABELS}")

print("=" * 70)

if TOTAL_IMAGES == TOTAL_LABELS:
    print("Dataset Ready for Training")
else:
    print("Warning: Image/Label count mismatch detected")

DATASET SUMMARY
Training Images    : 1688
Validation Images  : 161
Testing Images     : 80

Training Labels    : 1688
Validation Labels  : 161
Testing Labels     : 80

Total Images       : 1929
Total Labels       : 1929
Dataset Ready for Training


In [19]:
#  Training Configuration

TRAIN_CONFIG = {

    "data": DATA_YAML,

    "imgsz": IMAGE_SIZE,

    "epochs": EPOCHS,

    "batch": BATCH_SIZE,

    "workers": WORKERS,

    "device": DEVICE,

    "seed": SEED,

    "patience": 20,

    "optimizer": "auto",

    "lr0": LEARNING_RATE,

    "pretrained": True,

    "cache": False,

    "verbose": True

}

config_df = pd.DataFrame(

    TRAIN_CONFIG.items(),

    columns=["Parameter","Value"]

)

display(config_df)


,Parameter,Value
0,data,/content/drive/MyDrive/YOLO/data/SuspiciousBeh...
1,imgsz,640
2,epochs,50
3,batch,16
4,workers,2
5,device,0
6,seed,42
7,patience,20
8,optimizer,auto
9,lr0,0.001


In [20]:
#Verify Device

print("="*70)

print("Training Device")

print("="*70)

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

    print("CUDA :", torch.version.cuda)

    print("Device ID :", DEVICE)

else:

    print("CPU Training")

print("="*70)

Training Device
GPU : Tesla T4
CUDA : 12.8
Device ID : 0


In [21]:
#  Experiment Directories

YOLOV10_DIR = os.path.join(RESULT_DIR,"YOLOv10")

RTDETR_DIR = os.path.join(RESULT_DIR,"RTDETR")

os.makedirs(YOLOV10_DIR,exist_ok=True)

os.makedirs(RTDETR_DIR,exist_ok=True)

print("="*70)

print("Experiment Directories Created")

print("="*70)


print(YOLOV10_DIR)

print(RTDETR_DIR)

Experiment Directories Created
/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/results/YOLOv10
/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/results/RTDETR


In [22]:
# Model Configuration

MODELS = {

    "YOLOv10n":"yolov10n.pt",

    "RT-DETR":"rtdetr-l.pt"

}

model_df = pd.DataFrame(

    MODELS.items(),

    columns=["Model","Pretrained Weight"]

)

display(model_df)

,Model,Pretrained Weight
0,YOLOv10n,yolov10n.pt
1,RT-DETR,rtdetr-l.pt


In [23]:
#Training Ready

print("="*70)

print("Training Readiness Check")

print("="*70)

checks = {

    "Dataset YAML": os.path.exists(DATA_YAML),

    "Train Images": os.path.exists(TRAIN_IMAGES),

    "Train Labels": os.path.exists(TRAIN_LABELS),

    "Validation Images": os.path.exists(VALID_IMAGES),

    "Validation Labels": os.path.exists(VALID_LABELS),

    "Test Images": os.path.exists(TEST_IMAGES),

    "Test Labels": os.path.exists(TEST_LABELS)

}

for k,v in checks.items():

    print(f"{k:<22}: {'✓' if v else '✗'}")

print("="*70)

if all(checks.values()):

    print("Dataset is Ready for Training")

else:

    print("Dataset Verification Failed")

Training Readiness Check
Dataset YAML          : ✓
Train Images          : ✓
Train Labels          : ✓
Validation Images     : ✓
Validation Labels     : ✓
Test Images           : ✓
Test Labels           : ✓
Dataset is Ready for Training


In [24]:
# Training Summary

summary = pd.DataFrame({

    "Setting":[

        "Dataset",

        "Models",

        "Epochs",

        "Batch Size",

        "Image Size",

        "Optimizer",

        "Device"

    ],

    "Value":[

        DATA_YAML,

        " YOLOv10n | RT-DETR",

        EPOCHS,

        BATCH_SIZE,

        IMAGE_SIZE,

        "Auto",

        DEVICE

    ]

})

display(summary)

print("="*70)

print("Training Environment Ready")

print("="*70)

,Setting,Value
0,Dataset,/content/drive/MyDrive/YOLO/data/SuspiciousBeh...
1,Models,YOLOv10n | RT-DETR
2,Epochs,50
3,Batch Size,16
4,Image Size,640
5,Optimizer,Auto
6,Device,0


Training Environment Ready


YOLOv10n model


In [25]:
from ultralytics import YOLO

YOLO("yolov10n.pt")

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [26]:
# Load Pretrained YOLOv10n

from ultralytics import YOLO

MODEL_NAME = "yolov10n.pt"

yolov10_model = YOLO(MODEL_NAME)

print("=" * 70)
print("YOLOv10n Loaded Successfully")
print("=" * 70)

print("Model :", MODEL_NAME)

YOLOv10n Loaded Successfully
Model : yolov10n.pt


In [27]:
#  Model Information

print("=" * 70)
print("YOLOv10n Model Summary")
print("=" * 70)

print(yolov10_model.model)

YOLOv10n Model Summary
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inp

In [28]:
#Train YOLOv10n

PROJECT_NAME = "Suspicious_Behavior_Benchmark"
RUN_NAME = "YOLOv10n"

yolov10_results = yolov10_model.train(

    data=DATA_YAML,

    epochs=EPOCHS,

    imgsz=IMAGE_SIZE,

    batch=BATCH_SIZE,

    workers=WORKERS,

    device=DEVICE,

    optimizer="auto",

    project=PROJECT_NAME,

    name=RUN_NAME,

    exist_ok=True,

    seed=SEED,

    patience=20,

    cache=False,

    amp=True,

    val=True,

    plots=True,

    verbose=True
)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=YOLOv10n, nbs=64, nms=False, opset=None, optimize=F

In [29]:
# Save Best Model

import os

BEST_YOLOV10 = os.path.join(
    yolov10_results.save_dir,
    "weights",
    "best.pt"
)

LAST_YOLOV10 = os.path.join(
    yolov10_results.save_dir,
    "weights",
    "last.pt"
)

print("=" * 70)
print("YOLOv10n Training Completed")
print("=" * 70)

print("Best Model:")
print(BEST_YOLOV10)

print()

print("Last Model:")
print(LAST_YOLOV10)

print("=" * 70)

YOLOv10n Training Completed
Best Model:
/content/runs/detect/Suspicious_Behavior_Benchmark/YOLOv10n/weights/best.pt

Last Model:
/content/runs/detect/Suspicious_Behavior_Benchmark/YOLOv10n/weights/last.pt


In [30]:
#  Training Summary
summary = pd.DataFrame({

    "Model": ["YOLOv10n"],

    "Dataset": [os.path.basename(DATA_YAML)],

    "Epochs": [EPOCHS],

    "Image Size": [IMAGE_SIZE],

    "Batch Size": [BATCH_SIZE],

    "Optimizer": ["Auto"],

    "Device": [DEVICE]

})

display(summary)

print("=" * 70)
print("YOLOv10n Training Finished Successfully")
print("=" * 70)

,Model,Dataset,Epochs,Image Size,Batch Size,Optimizer,Device
0,YOLOv10n,data.yaml,50,640,16,Auto,0


YOLOv10n Training Finished Successfully


In [31]:
# Load Best YOLOv10n

from ultralytics import YOLO

best_yolov10 = YOLO(BEST_YOLOV10)

print("="*70)
print("Best YOLOv10n Loaded Successfully")
print("="*70)

Best YOLOv10n Loaded Successfully


In [32]:
#Evaluate

metrics = best_yolov10.val(

    data=DATA_YAML,

    split="test",

    imgsz=IMAGE_SIZE,

    batch=BATCH_SIZE,

    device=DEVICE,

    verbose=False

)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10n summary (fused): 102 layers, 2,265,753 parameters, 0 gradients, 6.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.8±0.4 ms, read: 29.9±10.8 MB/s, size: 68.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/YOLO/data/SuspiciousBehavior/test/labels... 80 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 124.4it/s 0.6s
val: New cache created: /content/drive/MyDrive/YOLO/data/SuspiciousBehavior/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.6it/s 3.1s
                   all         80        351      0.919      0.913      0.926      0.636
Speed: 8.1ms preprocess, 6.7ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val


In [33]:
# Extract Metrics

map50 = float(metrics.box.map50)

precision = float(metrics.box.mp)

recall = float(metrics.box.mr)

f1 = float(metrics.box.f1.mean())

print("=" * 70)

print(f"mAP@50    : {map50:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall    : {recall:.4f}")

print(f"F1-score  : {f1:.4f}")

print("=" * 70)

mAP@50    : 0.9255
Precision : 0.9189
Recall    : 0.9130
F1-score  : 0.9159


In [34]:
# Save Results
yolov10_results_df = pd.DataFrame({

    "Model": ["YOLOv10n"],

    "mAP@50": [map50],

    "Precision": [precision],

    "Recall": [recall],

    "F1-score": [f1]

})

display(yolov10_results_df)

YOLOV10_RESULT = os.path.join(

    RESULT_DIR,

    "YOLOv10_results.csv"

)

yolov10_results_df.to_csv(

    YOLOV10_RESULT,

    index=False

)

print("=" * 70)

print("YOLOv10 Results Saved Successfully")

print("=" * 70)

print(YOLOV10_RESULT)

,Model,mAP@50,Precision,Recall,F1-score
0,YOLOv10n,0.925516,0.918937,0.912986,0.915886


YOLOv10 Results Saved Successfully
/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/results/YOLOv10_results.csv


RT-DETR model

In [35]:
#  Load Pretrained RT-DETR

from ultralytics import RTDETR

MODEL_NAME = "rtdetr-l.pt"

rtdetr_model = RTDETR(MODEL_NAME)

print("=" * 70)
print("RT-DETR Loaded Successfully")
print("=" * 70)

print("Model :", MODEL_NAME)

RT-DETR Loaded Successfully
Model : rtdetr-l.pt


In [36]:
# Model Information

print("=" * 70)
print("RT-DETR Model Summary")
print("=" * 70)

print(rtdetr_model.model)

RT-DETR Model Summary
DetectionModel(
  (model): Sequential(
    (0): HGStem(
      (stem1): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (stem2a): Conv(
        (conv): Conv2d(32, 16, kernel_size=(2, 2), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (stem2b): Conv(
        (conv): Conv2d(16, 32, kernel_size=(2, 2), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
      )
      (stem3): Conv(
        (conv): Conv2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running

In [37]:
# Train RT-DETR

PROJECT_NAME = "Suspicious_Behavior_Benchmark"

RUN_NAME = "RTDETR"

rtdetr_results = rtdetr_model.train(

    data=DATA_YAML,

    epochs=4,

    imgsz=IMAGE_SIZE,

    batch=BATCH_SIZE,

    workers=WORKERS,

    device=DEVICE,

    optimizer="auto",

    project=PROJECT_NAME,

    name=RUN_NAME,

    exist_ok=True,

    seed=SEED,

    patience=2,

    cache=False,

    amp=True,

    val=True,

    plots=True,

    verbose=True

)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=4, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=RTDETR, nbs=64, nms=False, opset=None, optimize=Fals

In [38]:
#Save Best Model
import os

BEST_RTDETR = os.path.join(

    rtdetr_results.save_dir,

    "weights",

    "best.pt"

)

LAST_RTDETR = os.path.join(

    rtdetr_results.save_dir,

    "weights",

    "last.pt"

)

print("=" * 70)
print("RT-DETR Training Completed")
print("=" * 70)

print("Best Model:")
print(BEST_RTDETR)

print()

print("Last Model:")
print(LAST_RTDETR)

print("=" * 70)

RT-DETR Training Completed
Best Model:
/content/runs/detect/Suspicious_Behavior_Benchmark/RTDETR/weights/best.pt

Last Model:
/content/runs/detect/Suspicious_Behavior_Benchmark/RTDETR/weights/last.pt


In [39]:
#  Training Summary

summary = pd.DataFrame({

    "Model": ["RT-DETR"],

    "Dataset": [os.path.basename(DATA_YAML)],

    "Epochs": [EPOCHS],

    "Image Size": [IMAGE_SIZE],

    "Batch Size": [BATCH_SIZE],

    "Optimizer": ["Auto"],

    "Device": [DEVICE]

})

display(summary)

print("=" * 70)
print("RT-DETR Training Finished Successfully")
print("=" * 70)

,Model,Dataset,Epochs,Image Size,Batch Size,Optimizer,Device
0,RT-DETR,data.yaml,50,640,16,Auto,0


RT-DETR Training Finished Successfully


In [40]:
# Load Best RT-DETR Model

from ultralytics import RTDETR

best_rtdetr = RTDETR(BEST_RTDETR)

print("=" * 70)
print("Best RT-DETR Model Loaded Successfully")
print("=" * 70)

print(BEST_RTDETR)

Best RT-DETR Model Loaded Successfully
/content/runs/detect/Suspicious_Behavior_Benchmark/RTDETR/weights/best.pt


In [41]:
# Evaluate RT-DETR

metrics = best_rtdetr.val(

    data=DATA_YAML,

    split="test",

    imgsz=IMAGE_SIZE,

    batch=BATCH_SIZE,

    device=DEVICE,

    verbose=False

)

Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,989,905 parameters, 0 gradients, 103.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.3 ms, read: 41.1±18.2 MB/s, size: 68.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/YOLO/data/SuspiciousBehavior/test/labels.cache... 80 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 30.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.1s/it 5.5s
                   all         80        351      0.909      0.962      0.946      0.643
Speed: 8.9ms preprocess, 46.8ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/detect/val-2


In [42]:
# Extract Evaluation Metrics

map50 = float(metrics.box.map50)

precision = float(metrics.box.mp)

recall = float(metrics.box.mr)

f1 = float(metrics.box.f1.mean())

rtdetr_results_df = pd.DataFrame({

    "Model": ["RT-DETR"],

    "mAP@50": [map50],

    "Precision": [precision],

    "Recall": [recall],

    "F1-score": [f1]

})

display(rtdetr_results_df)

,Model,mAP@50,Precision,Recall,F1-score
0,RT-DETR,0.945503,0.909349,0.96152,0.934026


In [43]:
# Save Results

RTDETR_RESULT = os.path.join(

    RESULT_DIR,

    "RTDETR_results.csv"

)

rtdetr_results_df.to_csv(

    RTDETR_RESULT,

    index=False

)

print("=" * 70)

print("RT-DETR Results Saved Successfully")

print("=" * 70)

print(RTDETR_RESULT)

RT-DETR Results Saved Successfully
/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/results/RTDETR_results.csv


In [44]:
# Evaluation Summary

summary = rtdetr_results_df.copy()

display(summary)

print("=" * 70)

print("RT-DETR Evaluation Completed")

print("=" * 70)

,Model,mAP@50,Precision,Recall,F1-score
0,RT-DETR,0.945503,0.909349,0.96152,0.934026


RT-DETR Evaluation Completed


In [45]:
#  Load Evaluation Results

import os
import pandas as pd

YOLOV10_RESULT = os.path.join(
    RESULT_DIR,
    "YOLOv10_results.csv"
)

RTDETR_RESULT = os.path.join(
    RESULT_DIR,
    "RTDETR_results.csv"
)

yolov10_df = pd.read_csv(YOLOV10_RESULT)
rtdetr_df = pd.read_csv(RTDETR_RESULT)

print("=" * 70)
print("Evaluation Results Loaded Successfully")
print("=" * 70)

Evaluation Results Loaded Successfully


In [46]:
#Comparison Table

comparison = pd.concat(

    [

        yolov10_df,

        rtdetr_df

    ],

    ignore_index=True

)

comparison = comparison.round(4)

display(comparison)

,Model,mAP@50,Precision,Recall,F1-score
0,YOLOv10n,0.9255,0.9189,0.9130,0.9159
1,RT-DETR,0.9455,0.9093,0.9615,0.9340


In [47]:
# Model Ranking
comparison = comparison.sort_values(

    by="F1-score",

    ascending=False

).reset_index(drop=True)

comparison.insert(

    0,

    "Rank",

    range(1, len(comparison)+1)

)

display(comparison)

,Rank,Model,mAP@50,Precision,Recall,F1-score
0,1,RT-DETR,0.9455,0.9093,0.9615,0.9340
1,2,YOLOv10n,0.9255,0.9189,0.9130,0.9159


In [48]:
# Best Model

best_model = comparison.iloc[0]

print("=" * 70)

print("BEST DETECTION MODEL")

print("=" * 70)

print(f"Model      : {best_model['Model']}")
print(f"Precision  : {best_model['Precision']:.4f}")
print(f"Recall     : {best_model['Recall']:.4f}")
print(f"F1-score   : {best_model['F1-score']:.4f}")

print("=" * 70)

BEST DETECTION MODEL
Model      : RT-DETR
Precision  : 0.9093
Recall     : 0.9615
F1-score   : 0.9340


In [49]:
#  Save Final Comparison

FINAL_RESULT = os.path.join(

    RESULT_DIR,

    "Final_Model_Comparison.csv"

)

comparison.to_csv(

    FINAL_RESULT,

    index=False

)

print("=" * 70)

print("Final Comparison Saved Successfully")

print("=" * 70)

print(FINAL_RESULT)

Final Comparison Saved Successfully
/content/drive/MyDrive/YOLO/data/SuspiciousBehavior/results/Final_Model_Comparison.csv


In [50]:
# Benchmark Summary

display(comparison)

print("=" * 70)

print("Benchmark Completed Successfully")

print("=" * 70)

,Rank,Model,mAP@50,Precision,Recall,F1-score
0,1,RT-DETR,0.9455,0.9093,0.9615,0.9340
1,2,YOLOv10n,0.9255,0.9189,0.9130,0.9159


Benchmark Completed Successfully
